# Знаковая карта Шухарта (Shewhart Sign Chart)

**Непараметрическая контрольная карта для мониторинга положения процесса.**

Основано на главе 4 книги:

> Chakraborti, S. and Graham, M.A. (2019). *Nonparametric Statistical Process Control*, John Wiley & Sons Ltd.

---

## Содержание

1. [Импорты и настройка](#1-Импорты-и-настройка)
2. [Теория знаковой карты](#2-Теория-знаковой-карты-Шухарта)
    - 2.1. Постановка задачи
    - 2.2. Индикаторы Бернулли
    - 2.3. Знаковая статистика $SN_i$
    - 2.4. Распределение $T_i$ и $SN_i$ в IC
    - 2.5. Контрольные границы
    - 2.6. Distribution-free свойство
    - 2.7. Распределение длины серии
    - 2.8. Связь с классическим знаковым тестом
3. [Выбор целевого значения $\mu_0$](#3-Выбор-целевого-значения-μ0)
4. [Три подхода к расчёту константы $a$](#4-Три-подхода-к-расчёту-константы-a)
5. [Реализация функций](#5-Реализация-функций)
6. [Сравнение подходов](#6-Сравнение-подходов)
7. [Примеры 4.1–4.3 из книги](#7-Численные-примеры-из-книги)
8. [Моделирование длины серии](#8-Моделирование-длины-серии)
9. [Пользовательские данные data.csv](#9-Пользовательские-данные-datacsv)
10. [Итоги](#10-Итоги)

---

## 1. Импорты и настройка

In [ ]:
import sys
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import binom, geom, probplot
from scipy.special import comb

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['lines.linewidth'] = 1.5
pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 6)

# Корень проекта
PROJECT_ROOT = Path(os.getcwd()).resolve().parent
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path(os.getcwd()).resolve()
sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT = {PROJECT_ROOT}')

DATA_PATH = PROJECT_ROOT / 'data' / 'data.csv'
print(f'DATA_PATH = {DATA_PATH}, существует: {DATA_PATH.exists()}')

In [ ]:
# Принудительная перезагрузка модуля — на случай, если файл был обновлён
import importlib
try:
    import src.charts.sign_chart
    importlib.reload(src.charts.sign_chart)
    from src.charts.sign_chart import SignChart
    print('SignChart успешно импортирован (с reload)')
except Exception as e:
    print(f'Ошибка импорта: {e}')

---

## 2. Теория знаковой карты Шухарта

**Источник:** Section 4.2.1.1.1 «A Shewhart Sign Chart», стр. 189–196.

### 2.1. Постановка задачи *(стр. 189)*

Рассматривается процесс, из которого через равные промежутки времени берутся независимые подгруппы размера $n$:

$$X_{i1}, X_{i2}, \ldots, X_{in}, \quad i = 1, 2, 3, \ldots$$

**Предположения:**

1. Наблюдения **независимы** внутри подгруппы и между подгруппами.
2. Распределение $F$ **непрерывно** (но конкретный вид **не известен**).
3. Целевое значение $\mu_0$ (IC-медиана) **известно** — это Case K.

Мы хотим обнаружить **сдвиг медианы** от $\mu_0$ — вверх или вниз.

### 2.2. Индикаторы Бернулли *(стр. 189)*

Для каждого наблюдения определим индикатор:

$$Y_{ij} = \begin{cases} 1, & X_{ij} > \mu_0 \\ 0, & X_{ij} \le \mu_0 \end{cases}$$

**Ключевое наблюдение:** так как $\mu_0$ — медиана, а $F$ непрерывно:

$$P(X_{ij} > \mu_0) = P(X_{ij} < \mu_0) = 0.5, \quad P(X_{ij} = \mu_0) = 0$$

Следовательно, $Y_{ij} \sim \text{Bernoulli}(0.5)$ — **независимо от конкретного вида $F$**.

### 2.3. Знаковая статистика *(стр. 189–190, ур. 4.1)*

Определим функцию знака:

$$\phi(t) = \begin{cases} 1, & t > 0 \\ 0, & t = 0 \\ -1, & t < 0 \end{cases}$$

**Знаковая статистика:**

$$\boxed{SN_i = \sum_{j=1}^{n} \phi(X_{ij} - \mu_0) = n_i^+ - n_i^-}$$

где $n_i^+ = \sum_j Y_{ij}$ — число наблюдений выше $\mu_0$, $n_i^- = n - n_i^+$ — ниже.

**Эквивалентная форма:**

$$\boxed{SN_i = 2 n_i^+ - n = 2T_i - n}$$

где $T_i = n_i^+$ — **классическая знаковая статистика** (количество «+»).

### 2.4. Распределение $T_i$ и $SN_i$ в IC-состоянии *(стр. 192–193)*

**Теорема.** В IC-состоянии:

$$\boxed{T_i = \sum_{j=1}^{n} Y_{ij} \sim \text{BIN}(n, 0.5)}$$

**Доказательство.** $T_i$ — сумма $n$ независимых бернуллиевских величин с $p = 0.5$. По определению биномиального распределения, $T_i \sim \text{BIN}(n, 0.5)$. $\blacksquare$

**Моменты $T_i$:**

$$E(T_i) = np = \frac{n}{2}, \quad \text{Var}(T_i) = np(1-p) = \frac{n}{4}, \quad \text{SD}(T_i) = \frac{\sqrt{n}}{2}$$

**Моменты $SN_i = 2T_i - n$:**

$$\boxed{E(SN_i) = 2 \cdot \frac{n}{2} - n = 0}, \quad \boxed{\text{Var}(SN_i) = 4 \cdot \frac{n}{4} = n}, \quad \text{SD}(SN_i) = \sqrt{n}$$

**Симметрия.** $SN_i$ симметрично относительно $0$:

$$P(SN_i = k) = P(SN_i = -k), \quad \forall k \in \{-n, \ldots, n\}$$

Это прямое следствие симметрии биномиального распределения с $p = 0.5$.

**Формула PMF через $T_i$:**

$$P(SN_i = k) = P\left(T_i = \frac{n + k}{2}\right) = \binom{n}{(n+k)/2} 0.5^n$$

(определено, только если $n + k$ чётное).

### 2.5. Контрольные границы *(стр. 190, ур. 4.3)*

$$\boxed{UCL = +a, \quad CL = 0, \quad LCL = -a}$$

где $a$ — положительное целое, $0 < a \le n$.

**Правило сигнала:**

$$\text{Процесс OOC} \;\Longleftrightarrow\; |SN_i| \ge a \;\Longleftrightarrow\; SN_i \ge a \text{ или } SN_i \le -a$$

**Односторонние версии:**
- Верхняя: сигнал при $SN_i \ge a$ (только UCL).
- Нижняя: сигнал при $SN_i \le -a$ (только LCL).

### 2.6. Distribution-free свойство *(стр. 188)*

**Определение.** Контрольная карта называется **distribution-free**, если её распределение длины серии в IC-состоянии **одинаково для всех непрерывных распределений**.

**Теорема.** Знаковая карта Шухарта является distribution-free.

**Доказательство.**
1. $SN_i$ зависит только от знаков отклонений $\phi(X_{ij} - \mu_0)$.
2. Распределение $\phi(X_{ij} - \mu_0)$ определяется только знаком, а именно: $P(\phi = +1) = P(\phi = -1) = 0.5$ для **любого** непрерывного $F$.
3. Следовательно, распределение $SN_i$ (и длины серии $N$) **не зависит от $F$**. $\blacksquare$

**Практическое следствие:** $FAR$ и $ARL_{IC}$ знаковой карты одинаковы для нормального, экспоненциального, $t(4)$, гамма-распределения — для любого непрерывного распределения. Это **главное преимущество** знаковой карты перед параметрическими аналогами.

**Цена:** для приемлемого $ARL_{IC}$ требуется больший размер подгруппы — обычно $n \ge 10$, тогда как параметрические карты работают при $n = 5$.

### 2.7. Распределение длины серии *(стр. 193–194)*

**Определение.** Длина серии $N$ — число подгрупп, которые нужно собрать до **первого сигнала** (включая сигнальную).

**Теорема.** В Case K (параметры известны) длина серии имеет **геометрическое распределение**:

$$\boxed{N \sim \text{GEO}(\theta)}, \quad P(N = k) = (1-\theta)^{k-1} \theta, \quad k = 1, 2, 3, \ldots$$

где $\theta = FAR$ — вероятность сигнала в IC-состоянии.

**Доказательство.** В Case K:
1. Контрольные границы $\pm a$ фиксированы (не зависят от данных).
2. Наблюдения в разных подгруппах **независимы**.
3. Вероятность сигнала $P(|SN_i| \ge a)$ **одинакова** для всех $i$ и равна $\theta$.

Значит, каждая подгруппа — независимое испытание Бернулли: «сигнал» (успех) с вероятностью $\theta$, или «нет сигнала» с вероятностью $1-\theta$. $N$ — число испытаний до первого успеха. По определению, $N \sim \text{GEO}(\theta)$. $\blacksquare$

**Характеристики длины серии:**

| Величина | Формула |
|----------|---------|
| Средняя длина серии (ARL) | $ARL = E(N) = \dfrac{1}{\theta}$ |
| Дисперсия | $\text{Var}(N) = \dfrac{1-\theta}{\theta^2}$ |
| Стандартное отклонение (SDRL) | $SDRL = \dfrac{\sqrt{1-\theta}}{\theta}$ |
| Медиана (MRL) | $MRL = \left\lceil \dfrac{\ln 0.5}{\ln(1-\theta)} \right\rceil$ |
| $p$-й процентиль | $Q(p) = \left\lceil \dfrac{\ln(1-p)}{\ln(1-\theta)} \right\rceil$ |

**Замечания о свойствах геометрического распределения:**

1. **Правая асимметрия:** распределение сильно скошено вправо, особенно при малых $\theta$ (больших $ARL$).
2. **$SDRL \approx ARL$:** при малых $\theta$ имеем $SDRL \approx 1/\theta = ARL$. Вариабельность огромна.
3. **$MRL < ARL$:** из-за асимметрии медиана меньше среднего. Например, при $\theta = 0.0027$: $ARL = 370$, но $MRL \approx 256$.

**Практический вывод:** использование только $ARL$ может вводить в заблуждение. Рекомендуется дополнительно смотреть $MRL$ и другие процентили (Chakraborti, 2007).

### 2.8. Связь с классическим знаковым тестом *(стр. 190)*

Статистика $T_i = n_i^+$ — это основа **знакового теста** для проверки гипотезы:

$$H_0: \text{медиана} = \mu_0 \quad \text{против} \quad H_1: \text{медиана} \neq \mu_0$$

В IC-состоянии $T_i \sim \text{BIN}(n, 0.5)$, что позволяет вычислить $p$-значение точным образом.

**Связь:** знаковая карта — это **последовательное применение** знакового теста к каждой подгруппе.

**Преимущество карты:** она позволяет **мониторить процесс непрерывно**, а не проверять одну гипотезу.

---

## 3. Выбор целевого значения $\mu_0$

### 3.1. Проблема

Контрольные границы знаковой карты строятся относительно $\mu_0$ — целевого значения (IC-медианы). Возможны три ситуации:

1. **$\mu_0$ известно из инженерных соображений** (например, спецификация: диаметр должен быть 10 мм).
   - Используется напрямую.
   - В этом случае карта работает в **Case K** (стандарты известны).

2. **$\mu_0$ неизвестно, но есть Phase I (референсные данные)**.
   - $\mu_0$ оценивается как **медиана всех Phase I данных**.
   - Затем карта применяется к Phase II (новым подгруппам).
   - Это уже **Case U** — параметры оцениваются.

3. **$\mu_0$ неизвестно, Phase I нет**.
   - Используем медиану **всех имеющихся данных**.
   - Это компромисс: медиана — робастная оценка центра.

### 3.2. Почему медиана?

**Медиана** — естественный выбор для знаковой карты, потому что:

1. **Согласованность:** знаковая статистика $SN_i$ измеряет отклонение от медианы. Если использовать среднее вместо медианы, то $P(X_{ij} > \mu_0)$ уже **не будет** равно $0.5$ при асимметричных распределениях, и распределение $T_i$ перестанет быть $\text{BIN}(n, 0.5)$.

2. **Робастность:** медиана устойчива к выбросам, в отличие от среднего.

3. **Distribution-free:** при использовании медианы как $\mu_0$ карта остаётся distribution-free для **любого непрерывного распределения**.

### 3.3. Реализация в классе `SignChart`

```python
if target_value is None:
    self.target_value = self._calculate_median()
else:
    self.target_value = float(target_value)
```

Метод `_calculate_median`:

```python
def _calculate_median(self) -> float:
    all_values = self.data.iloc[:, :self.sample_size].values.flatten()
    all_values = all_values[~np.isnan(all_values)]
    return float(np.median(all_values)) if len(all_values) > 0 else 0.0
```

### 3.4. Ограничения

- Если в данных много выбросов или пропусков, медиана может быть смещена.
- В этом случае рекомендуется **явно задать $\mu_0$** (или использовать Phase I данные).
- Если данные явно **двумодальные**, медиана может попасть между модами — знаковая карта потеряет смысл.

### 3.5. Практическая рекомендация

**По умолчанию** используйте медиану всех данных. Если есть спецификация — задавайте $\mu_0$ явно. Если есть Phase I — считайте медиану **по Phase I**, а не по всем данным.

---

## 4. Три подхода к расчёту константы $a$

### 4.1. Постановка задачи

Константа $a$ определяет ширину контрольных границ $\pm a$. Она выбирается так, чтобы обеспечить желаемое значение $FAR$ (или $\alpha$).

**Ключевая формула для двусторонней карты:**

$$FAR = P(|SN_i| \ge a) = P(SN_i \ge a) + P(SN_i \le -a) = 2 \cdot P(SN_i \ge a)$$

Так как $SN_i = 2T_i - n$:

$$P(SN_i \ge a) = P\left(T_i \ge \left\lceil \frac{n+a}{2} \right\rceil\right) = \sum_{k=\lceil (n+a)/2 \rceil}^{n} \binom{n}{k} 0.5^n$$

### 4.2. Консервативный подход (по книге, стр. 191)

**Определение:**

$$\boxed{a = \text{наименьшее целое}: \; P(SN \ge a) \le \frac{FAR_0}{2}}$$

**Алгоритм:**
1. Для $a = n, n-1, \ldots, 1$ вычислить $P(SN \ge a)$.
2. Найти **наибольшее** $a$, для которого $P(SN \ge a) \le FAR_0/2$.
3. Это и есть $a$.

**Свойство:** гарантирует $FAR \le FAR_0$.

**Почему используется консервативный подход:** из-за дискретности $T_i$ точное равенство $P(SN \ge a) = FAR_0/2$ обычно **недостижимо**. Мы выбираем ближайшее **снизу**, чтобы гарантированно не превысить $FAR_0$.

**Промышленный стандарт:** $FAR_0 = 0.0027$ (соответствует 3-сигма-контролю).

### 4.3. Ближайший к $\alpha$

**Определение:**

$$\boxed{a = \arg\min_{a \in \{1, \ldots, n\}} |P(|SN| \ge a) - \alpha|}$$

**Алгоритм:**
1. Для каждого $a = 1, \ldots, n$ вычислить $P(|SN| \ge a)$.
2. Найти $a$, для которого абсолютная разность с $\alpha$ минимальна.

**Свойство:** достигнутый $FAR$ может **превышать** $\alpha$.

**Применение:** обучение, сравнение с классическим тестом, где $\alpha$ — точная вероятность ошибки I рода.

### 4.4. Симметричный квантильный

**Определение (для нечётных $n$):**

$$\boxed{a = \text{наименьшее нечётное}: \; P(SN \ge a) \le \frac{FAR_0}{2}}$$

Для чётных $n$ совпадает с консервативным подходом.

**Зачем:** для нечётных $n$ распределение $SN_i$ принимает **нечётные** значения: $\{-n, -n+2, \ldots, n-2, n\}$. Если $a$ чётное, то значения $SN_i = \pm(a-1)$ остаются «внутри», но несимметрично — это может искажать интерпретацию.

**Пример:** $n = 9$, $FAR_0 = 0.0027$.
- Консервативный может дать $a = 8$ (чётное).
- Симметричный даст $a = 9$ (нечётное, если это допустимо).

### 4.5. Сравнение подходов

| Подход | Формула | Гарантия $FAR \le FAR_0$ | Применение |
|--------|---------|--------------------------|------------|
| Консервативный | $P(SN \ge a) \le FAR_0/2$ | Да | Стандарт SPC (книга) |
| Ближайший | $|P(|SN| \ge a) - \alpha| \to \min$ | Нет | Обучение, сравнение |
| Симметричный | Наименьшее нечётное с $P(SN \ge a) \le FAR_0/2$ | Да | Нечётные $n$ |

---

## 5. Реализация функций

In [ ]:
def sign_statistic(sample, mu0):
    """SN_i = n_i⁺ − n_i⁻."""
    sample = np.asarray(sample)
    return int(np.sum(np.sign(sample - mu0)))


def p_sn_ge_a(n, a):
    """P(SN ≥ a) = P(T ≥ ceil((n+a)/2)), T ~ BIN(n, 0.5)."""
    if a > n: return 0.0
    if a <= -n: return 1.0
    t = int(np.ceil((n + a) / 2))
    if t > n: return 0.0
    if t < 0: return 1.0
    return float(np.sum([comb(n, m) * (0.5 ** n) for m in range(t, n + 1)]))


def p_abs_sn_ge_a(n, a):
    """P(|SN| ≥ a) = 2·P(SN ≥ a) по симметрии."""
    if a <= 0: return 1.0
    return min(2.0 * p_sn_ge_a(n, a), 1.0)


def calc_a_conservative(n, far0=0.0027):
    """Консервативный подход."""
    target = far0 / 2
    for a in range(n, 0, -1):
        if p_sn_ge_a(n, a) <= target:
            return a, 2.0 * p_sn_ge_a(n, a)
    return 1, 2.0 * p_sn_ge_a(n, 1)


def calc_a_nearest(n, alpha=0.05):
    """Ближайший к α."""
    best = (1, 1.0, float('inf'))
    for a in range(1, n + 1):
        p = p_abs_sn_ge_a(n, a)
        d = abs(p - alpha)
        if d < best[2]:
            best = (a, p, d)
    return best[0], best[1]


def calc_a_symmetric(n, far0=0.0027):
    """Симметричный квантильный."""
    if n % 2 == 0:
        return calc_a_conservative(n, far0)
    target = far0 / 2
    start = n if n % 2 == 1 else n - 1
    for a in range(start, 0, -2):
        if p_sn_ge_a(n, a) <= target:
            return a, 2.0 * p_sn_ge_a(n, a)
    return 1, 2.0 * p_sn_ge_a(n, 1)


def default_target_value(data, sample_size):
    """Медиана всех данных как целевое значение по умолчанию."""
    values = data.iloc[:, :sample_size].values.flatten()
    values = values[~np.isnan(values)]
    return float(np.median(values)) if len(values) > 0 else 0.0

---

## 6. Сравнение подходов

In [ ]:
def compare_all(n, far0=0.0027, alpha=0.05):
    a_c, far_c = calc_a_conservative(n, far0)
    a_n, far_n = calc_a_nearest(n, alpha)
    a_s, far_s = calc_a_symmetric(n, far0)
    return [
        {'approach': 'conservative', 'a': a_c, 'FAR': far_c,
         'ARL_IC': 1/far_c if far_c > 0 else np.inf},
        {'approach': 'nearest', 'a': a_n, 'FAR': far_n,
         'ARL_IC': 1/far_n if far_n > 0 else np.inf},
        {'approach': 'symmetric', 'a': a_s, 'FAR': far_s,
         'ARL_IC': 1/far_s if far_s > 0 else np.inf},
    ]

rows = []
for n in [5, 8, 10, 13, 14, 20, 25, 30]:
    for r in compare_all(n):
        rows.append({'n': n, **r})
df_cmp = pd.DataFrame(rows)
df_cmp.pivot(index='n', columns='approach', values=['a', 'FAR', 'ARL_IC']).round(4)

In [ ]:
n_range = np.arange(2, 31)
data_plot = {'conservative': ([], []), 'nearest': ([], []), 'symmetric': ([], [])}
for n in n_range:
    data_plot['conservative'][0].append(calc_a_conservative(n)[0])
    data_plot['conservative'][1].append(calc_a_conservative(n)[1])
    data_plot['nearest'][0].append(calc_a_nearest(n, 0.05)[0])
    data_plot['nearest'][1].append(calc_a_nearest(n, 0.05)[1])
    data_plot['symmetric'][0].append(calc_a_symmetric(n)[0])
    data_plot['symmetric'][1].append(calc_a_symmetric(n)[1])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = {'conservative': 'steelblue', 'nearest': 'crimson', 'symmetric': 'forestgreen'}
styles = {'conservative': 'o-', 'nearest': 's--', 'symmetric': '^:'}

for key, label in [('conservative', 'Консервативный'),
                    ('nearest', 'Ближайший (α=0.05)'),
                    ('symmetric', 'Симметричный')]:
    axes[0].plot(n_range, data_plot[key][0], styles[key], color=colors[key], label=label)
    axes[1].plot(n_range, data_plot[key][1], styles[key], color=colors[key], label=label)

axes[0].set_xlabel('n'); axes[0].set_ylabel('a')
axes[0].set_title('Значения a'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].axhline(0.0027, color='steelblue', linestyle=':', alpha=0.6)
axes[1].axhline(0.05, color='crimson', linestyle=':', alpha=0.6)
axes[1].set_xlabel('n'); axes[1].set_ylabel('FAR')
axes[1].set_title('Достигнутый FAR'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_yscale('log')
plt.tight_layout(); plt.show()

---

## 7. Численные примеры из книги

### 7.1. Пример 4.1 — стр. 219–221

30 наблюдений: 20 из $N(10,1)$, 10 из $N(11,1)$. $\mu_0 = 10$.

In [ ]:
data_4_5 = np.array([
    9.45, 7.99, 9.29, 11.66, 12.16, 10.18, 8.04, 11.46, 9.20, 10.34,
    9.03, 11.47, 10.51, 9.40, 10.08, 9.37, 10.62, 10.31, 8.52, 10.84,
    10.90, 9.33, 12.29, 11.50, 10.60, 11.08, 10.38, 11.62, 11.31, 10.52,
])
mu0, n = 10.0, 1
a, far = calc_a_conservative(n, 0.0027)
sn = np.array([sign_statistic([x], mu0) for x in data_4_5])
print(f'n={n}, a={a}, FAR={far:.4f}, ARL_IC={1/far:.2f}')

fig, ax = plt.subplots()
ax.plot(range(1, 31), sn, 'o-', color='steelblue')
ax.axhline(a, color='red', ls='--', label=f'UCL=+{a}')
ax.axhline(-a, color='red', ls='--', label=f'LCL=−{a}')
ax.axhline(0, color='green', ls='-', label='CL=0')
ax.set_title(f'Пример 4.1 (стр. 219–221): n={n}')
ax.set_xlabel('Наблюдение'); ax.set_ylabel('$SN_i$')
ax.legend(); ax.grid(alpha=0.3); plt.show()

### 7.2. Пример 4.2 — стр. 222–223

15 подгрупп по $n = 10$. $\mu_0 = 0$.

In [ ]:
data_4_6 = np.array([
    [-0.30, -1.28, 0.24, 1.28, 1.20, 1.73, -2.18, -0.23, 1.10, -1.09],
    [-0.69, -1.69, -1.85, -0.98, -0.77, -2.12, -0.57, -0.40, 0.13, -0.37],
    [-0.33, -0.37, 1.34, -0.09, -0.19, -0.51, 1.97, 0.87, 2.38, -0.65],
    [1.66, -1.61, 0.54, 0.90, 1.92, -0.08, -0.52, 0.68, -0.38, 0.76],
    [-1.44, -0.85, -1.52, -0.36, -0.03, 0.03, -0.32, 2.19, -1.74, -0.74],
    [-2.58, 1.45, -1.28, -0.65, 0.76, 0.47, 0.87, 0.60, -1.37, -1.12],
    [0.69, 0.32, -0.94, -0.24, 0.13, 0.56, 0.14, -0.91, 1.88, 0.49],
    [0.07, 0.83, 0.86, -0.64, -0.92, 1.11, -1.20, -1.56, 0.71, 0.64],
    [2.21, 1.44, 1.30, 0.11, 0.00, 0.45, -0.03, -1.05, -1.77, 0.83],
    [0.44, 0.62, 0.21, -1.03, 1.24, -0.31, -0.84, -0.82, -0.43, -0.45],
    [-0.52, 0.85, 0.51, -0.61, 1.30, -1.76, 0.55, -0.12, 0.04, -0.65],
    [-0.55, 0.85, 0.80, 0.46, 0.69, 1.63, 0.30, 0.59, 1.85, -0.34],
    [1.04, 0.14, 1.14, -0.15, -0.78, 1.08, -0.58, 0.53, 0.55, -0.32],
    [-0.44, -1.37, 1.99, -0.57, 0.09, -0.23, 2.84, 1.25, 0.88, 1.33],
    [0.19, 0.54, -0.25, -1.22, 1.27, -0.29, -1.31, 0.76, 0.78, 0.43],
])
mu0, n = 0.0, 10
a, far = calc_a_conservative(n, 0.0027)
sn = np.array([sign_statistic(row, mu0) for row in data_4_6])

print(f'n={n}, a={a}, FAR={far:.6f}, ARL_IC={1/far:.2f}')
fig, ax = plt.subplots()
ax.plot(range(1, 16), sn, 'o-', color='steelblue', markersize=7)
ax.axhline(a, color='red', ls='--', label=f'UCL=+{a}')
ax.axhline(-a, color='red', ls='--', label=f'LCL=−{a}')
ax.axhline(0, color='green', ls='-', label='CL=0')
ax.set_title(f'Пример 4.2 (стр. 222–223): n={n}')
ax.set_xlabel('Номер подгруппы'); ax.set_ylabel('$SN_i$')
ax.legend(); ax.grid(alpha=0.3); plt.show()

### 7.3. Пример 4.3 — стр. 224–225

Правило 2-of-2 с предупреждающими границами $UWL=+6, LWL=-6$.

In [ ]:
UWL, LWL = 6, -6
signals = []
for i in range(len(sn)):
    if abs(sn[i]) >= a: signals.append(('1-of-1', i+1, sn[i]))
    if i >= 1:
        prev, cur = sn[i-1], sn[i]
        if (prev >= UWL and cur >= UWL) or (prev <= LWL and cur <= LWL):
            signals.append(('2-of-2', i+1, (prev, cur)))
print('Сигналы:', signals if signals else 'нет')

fig, ax = plt.subplots()
ax.plot(range(1, 16), sn, 'o-', color='steelblue', markersize=7)
for lvl, c, lbl in [(a, 'red', f'UCL={a}'), (-a, 'red', f'LCL=−{a}'),
                     (UWL, 'orange', f'UWL={UWL}'), (LWL, 'orange', f'LWL={LWL}'),
                     (0, 'green', 'CL=0')]:
    ax.axhline(lvl, color=c, ls='--', alpha=0.8, label=lbl)
ax.set_title('Пример 4.3 (стр. 224–225): 2-of-2'); ax.legend(fontsize=9)
ax.grid(alpha=0.3); plt.show()

---

## 8. Моделирование длины серии

**Источник:** Section 4.2.1.1.3, стр. 195–196.

In [ ]:
# def simulate_rl(n, mu0=0.0, a=None, n_sims=20000):
#     if a is None:
#         a, _ = calc_a_conservative(n)
#     rls = np.empty(n_sims, dtype=int)
#     for sim in range(n_sims):
#         t = 0
#         while True:
#             t += 1
#             sample = np.random.normal(mu0, 1.0, size=n)
#             if abs(sign_statistic(sample, mu0)) >= a:
#                 break
#         rls[sim] = t
#     return rls

# n_sim = 10
# a_sim, far_sim = calc_a_conservative(n_sim)
# rl = simulate_rl(n_sim, 0.0, a_sim, 20000)
# print(f'n={n_sim}, a={a_sim}')
# print(f'ARL: {rl.mean():.2f} vs теория {1/far_sim:.2f}')
# print(f'MRL: {np.median(rl):.0f} vs теория {int(np.ceil(np.log(0.5)/np.log(1-far_sim)))}')
# print(f'SDRL: {rl.std():.2f} vs теория {np.sqrt(1-far_sim)/far_sim:.2f}')

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# axes[0].hist(rl, bins=80, color='steelblue', alpha=0.8, density=True, edgecolor='white')
# x = np.arange(1, rl.max()+1)
# axes[0].plot(x, geom.pmf(x, far_sim), 'r-', lw=2, label=f'GEO(FAR={far_sim:.4f})')
# axes[0].set_xlabel('N'); axes[0].set_ylabel('Плотность')
# axes[0].set_title('Распределение длины серии'); axes[0].legend(); axes[0].grid(alpha=0.3)
# probplot(rl, dist=geom(far_sim), plot=axes[1]); axes[1].grid(alpha=0.3)
# plt.tight_layout(); plt.show()

---

## 9. Пользовательские данные `data.csv`

**Файл:** `data/data.csv` — 10 строк × 14 столбцов (10 подгрупп по $n = 14$).

**Целевое значение по умолчанию:** медиана всех данных (согласно логике класса).

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Файл не найден: {DATA_PATH}')
df = pd.read_csv(DATA_PATH)
print('Первые 5 строк:'); display(df.head())
print(f'Размер: {df.shape}')
display(df.describe())

### 9.1. Целевое значение по умолчанию — медиана всех данных

In [ ]:
n_obs = df.shape[1]  # 14 столбцов = 14 наблюдений в подгруппе
median_all = default_target_value(df, n_obs)
print(f'n = {n_obs}')
print(f'Медиана всех данных (μ₀ по умолчанию): {median_all:.6f}')
print(f'Среднее всех данных: {df.values.flatten().mean():.6f}')
print(f'Мин / Макс: {df.values.flatten().min():.4f} / {df.values.flatten().max():.4f}')

### 9.2. Сравнение подходов для текущего $n$

In [ ]:
cmp_rows = []
for approach, label in [('conservative', 'Консервативный'),
                         ('nearest', 'Ближайший'),
                         ('symmetric', 'Симметричный')]:
    if approach == 'conservative':
        a_, far_ = calc_a_conservative(n_obs, 0.0027)
    elif approach == 'nearest':
        a_, far_ = calc_a_nearest(n_obs, 0.05)
    else:
        a_, far_ = calc_a_symmetric(n_obs, 0.0027)
    cmp_rows.append({'Подход': label, 'a': a_, 'FAR': far_,
                     'ARL_IC': 1/far_ if far_ > 0 else np.inf})
df_n_cmp = pd.DataFrame(cmp_rows).round(6)
display(df_n_cmp)

### 9.3. Класс `SignChart` — консервативный подход (μ₀ = медиана по умолчанию)

In [ ]:
chart_cons = SignChart(df, sample_size=n_obs, 
                       approach='conservative', far0=0.0027,
                       coefficients_file=f'{PROJECT_ROOT}/src/utils/coefficients.json')
lims = chart_cons.calculate_limits()
print(f'Использованное μ₀ = {chart_cons.target_value:.6f}')
print('Контрольные границы:')
for k in ['CL', 'UCL', 'LCL', 'c_value', 'FAR', 'ARL_IC', 'MRL_IC']:
    print(f'  {k} = {lims[k]}')
print('\nТаблица сравнения подходов:')
display(chart_cons.comparison_table)
print('\nСводка:')
display(chart_cons.get_summary())
print('\nЗначения SN:')
display(chart_cons.get_sn_table())

fig, ax = chart_cons.plot(fill_zones=True)
plt.show()

### 9.4. Класс `SignChart` — ближайший подход

In [ ]:
chart_near = SignChart(df, sample_size=n_obs, 
                       approach='nearest', alpha=0.05,
                       coefficients_file=f'{PROJECT_ROOT}/src/utils/coefficients.json')
chart_near.calculate_limits()
print(f'μ₀ = {chart_near.target_value:.6f}, a = {chart_near.control_limits["c_value"]}')
display(pd.DataFrame([chart_near.get_summary()]))

fig, ax = chart_near.plot(fill_zones=True)
plt.show()

### 9.5. График сравнения подходов на диапазоне $n$

In [ ]:
fig, axes = chart_cons.plot_comparison(n_range=np.arange(2, 31), figsize=(14, 9))
plt.show()

### 9.6. Прямое сравнение границ на `data.csv`

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sn_vals = chart_cons.subgroups['sn'].values
x = chart_cons.subgroups['subgroup'].values
ax.plot(x, sn_vals, 'o-', color='steelblue', markersize=8, label='$SN_i$')

a_cons = chart_cons.control_limits['c_value']
a_near = chart_near.control_limits['c_value']
ax.axhline(a_cons, color='red', ls='--', lw=1.5, label=f'Консервативный: a={a_cons}')
ax.axhline(-a_cons, color='red', ls='--', lw=1.5)
ax.axhline(a_near, color='orange', ls=':', lw=1.5, label=f'Ближайший: a={a_near}')
ax.axhline(-a_near, color='orange', ls=':', lw=1.5)
ax.axhline(0, color='green', ls='-', alpha=0.5, label='CL=0')
ax.set_xlabel('Номер подгруппы'); ax.set_ylabel('$SN_i$')
ax.set_title(f'Сравнение подходов на data.csv (n={n_obs}, μ₀={chart_cons.target_value:.4f})')
ax.legend(); ax.grid(alpha=0.3); plt.show()

### 9.7. Явное задание μ₀ для сравнения

In [ ]:
# Сравнение: медиана vs среднее vs 0
chart_median = SignChart(df, sample_size=n_obs, target_value=median_all,
                       coefficients_file=f'{PROJECT_ROOT}/src/utils/coefficients.json')
chart_mean = SignChart(df, sample_size=n_obs, target_value=df.values.flatten().mean(),
                       coefficients_file=f'{PROJECT_ROOT}/src/utils/coefficients.json')
chart_zero = SignChart(df, sample_size=n_obs, target_value=0.0,
                       coefficients_file=f'{PROJECT_ROOT}/src/utils/coefficients.json')

for name, ch in [('медиана', chart_median), ('среднее', chart_mean), ('0', chart_zero)]:
    ch.calculate_limits()
    sn = ch.subgroups['sn'].values
    print(f'μ₀ = {ch.target_value:>10.4f} ({name}): SN = {sn.tolist()}, sum = {sn.sum()}')

**Наблюдение:** при $\mu_0 = $ медиана сумма $SN$ близка к 0 — это следствие симметрии. При других $\mu_0$ сумма сдвигается.

---

## 10. Итоги

### 10.1. Ключевые формулы

| Величина | Формула | Источник |
|----------|---------|----------|
| Индикатор | $Y_{ij} = \mathbb{1}(X_{ij} > \mu_0)$ | стр. 189 |
| Знаковая статистика | $SN_i = n_i^+ - n_i^- = 2T_i - n$ | ур. 4.1, стр. 189 |
| Распределение в IC | $T_i \sim \text{BIN}(n, 0.5)$ | стр. 192 |
| Моменты | $E(SN_i) = 0$, $\text{Var}(SN_i) = n$ | стр. 193 |
| Границы | $LCL = -a$, $CL = 0$, $UCL = +a$ | ур. 4.3, стр. 190 |
| Правило сигнала | $|SN_i| \ge a$ | стр. 190 |
| Длина серии | $N \sim \text{GEO}(FAR)$ | стр. 193 |
| ARL | $1/FAR$ | стр. 193 |
| SDRL | $\sqrt{1-FAR}/FAR$ | стр. 193 |
| MRL | $\lceil \ln 0.5 / \ln(1-FAR) \rceil$ | стр. 194 |

### 10.2. Три подхода к $a$

| Подход | Формула | Гарантия |
|--------|---------|----------|
| Консервативный | $P(SN \ge a) \le FAR_0/2$ | $FAR \le FAR_0$ |
| Ближайший | $\|P(\|SN\| \ge a) - \alpha\| \to \min$ | Нет |
| Симметричный | Наименьшее нечётное с $P(SN \ge a) \le FAR_0/2$ | $FAR \le FAR_0$ |

### 10.3. Целевое значение $\mu_0$

- **По умолчанию:** медиана всех данных — робастная и согласованная оценка.
- **Если есть спецификация:** используйте её явно.
- **Если есть Phase I:** используйте медиану Phase I данных.

### 10.4. Практические рекомендации

1. **Консервативный подход** — стандарт SPC (книга, стр. 191).
2. **Знаковая карта** требует $n \ge 10$ для приемлемого $ARL_{IC}$.
3. Для $n = 1$ используйте **CUSUM-SN** или **EWMA-SN** (стр. 221).
4. **IC-робастность** — главное преимущество карты (стр. 188).
5. **Для `data.csv`** ($n = 14$) все три подхода дают близкие значения $a$.